# 022 — Sistemas expertos y motores de reglas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("logic", seed=22)
assert result["kind"] == "logic"
assert result["evidence"]
show(result)


## Solución 1 — Punto fijo

```text
Ciclo 1: R1 dispara (sus 2 premisas son hechos iniciales) → puede_experimentar
Ciclo 2: R2 dispara → requiere_baseline
Ciclo 3: R3 dispara → requiere_evaluacion
Ciclo 4: ninguna regla nueva → punto fijo
```

`facts` (ordenado alfabéticamente, como lo emite el laboratorio) =
`[puede_experimentar, requiere_baseline, requiere_evaluacion, tiene_datos,
tiene_objetivo]`; `rules_fired` tiene 3 entradas en el orden R1, R2, R3.


In [ ]:
result = run_lab("logic", seed=22)
assert result["result"]["facts"] == sorted([
    "tiene_datos", "tiene_objetivo", "puede_experimentar",
    "requiere_baseline", "requiere_evaluacion"])
assert [f["then"] for f in result["result"]["rules_fired"]] == [
    "puede_experimentar", "requiere_baseline", "requiere_evaluacion"]
print("predicción verificada ✔")


## Solución 2 — Base extendida

a) R4 necesita `requiere_evaluacion` (derivado en el ciclo 3) y `tiene_riesgo`
(inicial): dispara en el ciclo 4 → punto fijo con **6 hechos** y 4 reglas
disparadas.

c) **No**: el orden de evaluación puede cambiar el orden de `fired`, pero no el
punto fijo — el forward chaining sobre Horn es confluente: mientras el bucle
siga hasta que nada cambie, el conjunto final de hechos es el mismo. (En
motores reales con acciones que *retractan* hechos, el orden SÍ importa: por
eso existen estrategias de resolución de conflicto.)


In [ ]:
facts = {"tiene_datos", "tiene_objetivo", "tiene_riesgo"}
rules = [
    ({"tiene_datos", "tiene_objetivo"}, "puede_experimentar"),
    ({"puede_experimentar"}, "requiere_baseline"),
    ({"requiere_baseline"}, "requiere_evaluacion"),
    ({"requiere_evaluacion", "tiene_riesgo"}, "requiere_supervision"),
]
fired = []
changed = True
while changed:
    changed = False
    for cond, concl in rules:
        if cond <= facts and concl not in facts:
            facts.add(concl)
            fired.append(concl)
            changed = True
assert "requiere_supervision" in facts and len(fired) == 4
print("punto fijo extendido:", sorted(facts))


## Solución 3 — Backward chaining

```text
objetivo: requiere_supervision
  ← R4 exige: requiere_evaluacion  Y  tiene_riesgo (hecho inicial ✔)
     objetivo: requiere_evaluacion
       ← R3 exige: requiere_baseline
          ← R2 exige: puede_experimentar
             ← R1 exige: tiene_datos ✔  y  tiene_objetivo ✔
```

Backward derivó exactamente los mismos 4 hechos intermedios que forward — en
esta base lineal no hay ahorro. El ahorro aparece cuando la base tiene reglas
irrelevantes para el objetivo: forward las dispararía todas; backward ni las
mira. Moral: la dirección se elige por la forma de la consulta y de la base,
no por corrección.


## Solución 4 — Factores de certeza

a) `CF_a = 0,7 × 0,6 = 0,42`; `CF_b = 0,5 × 1,0 = 0,50`;
combinado: `0,42 + 0,50 × (1 − 0,42) = 0,42 + 0,29 = 0,71`.

b) No: la fórmula es asintótica a 1 — cada evidencia nueva cubre una fracción
de la incertidumbre restante. Es deliberado: ninguna cantidad de evidencia
imperfecta debe producir certeza absoluta. El supuesto oculto es la
**independencia** de las evidencias: si ambas reglas derivan del mismo síntoma,
0,71 sobreestima (se cuenta la misma información dos veces) — la limitación que
las redes bayesianas (parte 02) resuelven con estructura explícita.


In [ ]:
cf_a = 0.7 * 0.6
cf_b = 0.5 * 1.0
cf_comb = cf_a + cf_b * (1 - cf_a)
assert abs(cf_comb - 0.71) < 1e-9
print(f"CF_a={cf_a:.2f}, CF_b={cf_b:.2f}, combinado={cf_comb:.2f} ✔")


## Reflexión

1. El laboratorio responde '¿qué se deriva de estos datos?' (forward). Formula la consulta inversa ('¿se requiere evaluación?') y describe la traza de backward chaining: ¿qué subobjetivos genera y en qué orden?
2. Si la base tuviera 10 000 reglas y llegara un hecho nuevo por segundo, ¿por qué el MATCH ingenuo colapsa y qué hace exactamente Rete para que el costo dependa del cambio y no del total?
3. Dos reglas independientes concluyen lo mismo con CF 0,6 y 0,5. La combinación da 0,8. ¿Qué supuesto oculto hay en ese número y qué pasaría si ambas reglas se basaran en el mismo síntoma?
